# Vendor Master Data Quality

Initial scope: SAP-style vendor master data.

Phase 1 focus:
- General vendor data
- Data profiling
- Deterministic Data Quality rules
- Duplicate and similarity analysis later

In [64]:
import pandas as pd
vendor_columns = [
    "LIFNR",
    "LAND1",
    "NAME1",
    "NAME2",
    "ORT01",
    "PSTLZ",
    "STRAS",
    "TELF1",
    "STCD1",
    "SPERR",
    "LOEVM"
]

len(vendor_columns)

11

In [65]:
vendor_data = {
    "LIFNR": [
        "100001", "100002", "100003", "100004", "100005",
        "100006", "100007", "100008", "100009", "100010",
        "100011", "100012"
    ],

"KTOKK": [
    "ZSUP", "ZSUP", "ZSUP", "ZSUP", "ZSUP",
    "ZSUP", "ZSUP", "ZSER", "ZSUP", "ZSER",
    "ZSUP", "ZSUP"
],

    "LAND1": [
        "AE", "AE", "AE", "OM", "AE",
        "AE", "AE", None, "AE", "OM",
        "AE", "AE"
    ],

    "NAME1": [
        "Gulf Trading LLC",
        "Gulf Trading L.L.C.",
        "Al Noor Services",
        "Muscat Industrial Co",
        None,
        "Desert Engineering LLC",
        "DESERT ENGINEERING L.L.C test",
        "Emirates Technical Services NA",
        "Al Falah NA Trading",
        "Oman Equipment Services do not use",
        "Future Tech Solutions",
        "Future Tech Solutions LLC"
    ],

    "NAME2": [
        None,
        None,
        "Maintenance Division",
        None,
        None,
        None,
        None,
        None,
        "Abu Dhabi Branch",
        None,
        None,
        None
    ],

    "ORT01": [
        "Abu Dhabi",
        "Abu Dhabi",
        "Dubai",
        "Muscat",
        "Dubai",
        "Abu Dhabi",
        "ABU DHABI",
        "Sharjah",
        "AbuDhabi",
        "Muscat",
        "Dubai",
        "Dubai"
    ],

    "PSTLZ": [
        "12345",
        "12345",
        None,
        "112",
        "00000",
        "45678",
        "45678",
        "54321",
        "12A45",
        "113",
        "67890",
        "67890"
    ],

    "STRAS": [
        "Mussafah Industrial Area",
        "Mussafah Ind. Area",
        "Sheikh Zayed Road",
        "Ruwi Industrial Estate",
        None,
        "Street 10, Mussafah",
        "Street 10 Mussafah",
        "Industrial Area 4",
        "Hamdan Street",
        "Ghala Industrial Area",
        "Business Bay Tower 2",
        "Business N/A, Tower 2"
    ],

    "TELF1": [
        "+971501234567",
        "0501234567",
        "+971 55 222 3344",
        "+96899112233",
        None,
        "02-5556677",
        "+97125556677",
        "12345",
        "+971501111222",
        "+968 24 567890",
        "+971504445555",
        "0504445555"
    ],

    "STCD1": [
        "100234567890003",
        "100234567890003",
        "100345678900003",
        "OM1234567",
        None,
        "100456789010003",
        "100456789010003",
        "ABC123",
        "100567890120003",
        "OM9876543",
        "100678901230003",
        "100678901230004"
    ],

    "SPERR": [
        "", "", "", "", "",
        "", "X", "", "", "",
        "", ""
    ],

    "LOEVM": [
        "", "", "", "", "",
        "", "", "", "X", "",
        "", ""
    ]
}


raw_data = pd.DataFrame(vendor_data)
#print(vendor_raw.info())
data = raw_data.copy()

rule_catalog=pd.read_excel(r'/Users/vivek/Documents/Data Science Projects/data-quality-solution/DQ_Rule_Catalog.xlsx')
dq_results = pd.DataFrame( columns=["WORKSTREAM","OBJECT","OBJECT_KEY","OBJECT_KEY_VALUE","RULE_ID","DQ_DIMENSION","FIELD","ISSUE"])


In [66]:

def add_dq_results(data,dq_results, condition, workstream, object , key, field, rule_id, dq_dimension, issue):
    rule_results = data[condition][[key]].copy()
    #print(len(rule_results))
    if(len(rule_results)>0):
        rule_results.rename(columns={key: "OBJECT_KEY_VALUE"}, inplace=True)
        rule_results["WORKSTREAM"]=workstream
        rule_results["OBJECT"]=object
        rule_results["OBJECT_KEY"]=key
        rule_results["RULE_ID"]=rule_id
        rule_results["DQ_DIMENSION"] = dq_dimension
        rule_results["FIELD"] = field
        rule_results["ISSUE"]= issue
        dq_results = pd.concat([dq_results, rule_results], ignore_index = True)

    #print(rule_results)
    
    dq_results = dq_results.drop_duplicates(
            subset=[
                "WORKSTREAM",
                "OBJECT",
                "OBJECT_KEY_VALUE",
                "RULE_ID"
            ],
            keep="last"
        )

    return dq_results
    #print(dq_results)


In [67]:
# Log for Standard Rule Category 
standard_missing = rule_catalog[ (rule_catalog["RULE_CLASS"]=="STANDARD") & (rule_catalog["RULE_TYPE"]=="MISSING")  ]

#standard_missing
for row in standard_missing.itertuples():
    condition = (data[row.FIELD].isna())  | (data[row.FIELD].str.strip()=='')
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)
    
#dq_results


In [68]:
# Logic for PLACEHOLDER_OR_SUSPICIOUS_TEXT category
import re

placeholder_values = [
    "TEST",
    "DUMMY",
    "XXX",
    "N/A",
    "NA",
    "UNKNOWN",
    "TBD",
    "TEMP",
    "SAMPLE",
    "DELETE",
    "DELETED",
    "REMOVE",
    "REMOVED",
    "NOT REQUIRED",
    "NOT APPLICABLE",
    "NOT NEEDED",
    "DO NOT USE",
    "DONT USE",
    "OBSOLETE"
]


pattern = r"\b(" + "|".join(
    re.escape(value) for value in placeholder_values
) + r")\b"

data[data["STRAS"].str.contains(pattern,case=False,na=False,regex=True)]

SUSPICIOUS_TEXT = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="PLACEHOLDER_OR_SUSPICIOUS_TEXT")  ]

# PLACEHOLDER_OR_SUSPICIOUS_TEXT
for row in SUSPICIOUS_TEXT.itertuples():
    condition = data[row.FIELD].str.contains(pattern,case=False,na=False,regex=True)
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)



/var/folders/5b/b99k41_n0sdg438t9w08svk00000gn/T/ipykernel_1449/1478555724.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data[data["STRAS"].str.contains(pattern,case=False,na=False,regex=True)]
/var/folders/5b/b99k41_n0sdg438t9w08svk00000gn/T/ipykernel_1449/1478555724.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  condition = data[row.FIELD].str.contains(pattern,case=False,na=False,regex=True)


In [69]:
# Logic for PHONE_VALIDATION 

PHONE_VALIDATION = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="PHONE_VALIDATION_BY_COUNTRY")  ]

# PHONE_VALIDATION
for row in PHONE_VALIDATION.itertuples():
    condition = ~data[row.FIELD].str.fullmatch(r"\+?\s?[0-9][0-9\s\-()]+",na=False)
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)


In [70]:
# Logic for POSTAL CODE VALIDATION 

POSTAL_CODE_VALIDATION = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="POSTAL_FORMAT_BY_COUNTRY")  ]

# POSTAL_CODE_VALIDATION
for row in POSTAL_CODE_VALIDATION.itertuples():
    condition = ~data[row.FIELD].str.fullmatch(r"[a-zA-Z0-9\s-]+",na=False)
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)


In [71]:
# Logic for TAX CODE VALIDATION 

TAX_CODE_VALIDATION = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="TAX_FORMAT_BY_COUNTRY")  ]

country_list = data["LAND1"].unique()
#print(country_list)


for country in country_list:
    if country=='AE':
        pattern = r"[0-9]{15}"
    elif country=='OM':
        pattern = r"[A-Za-z0-9]{12}"
    country_data = data[data["LAND1"]==country]
    

# TAX_CODE_VALIDATION
    for row in TAX_CODE_VALIDATION.itertuples():
        condition = ~country_data[row.FIELD].str.fullmatch(pattern,na=False)
        dq_results = add_dq_results(country_data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)


In [72]:
# General-purpose word -> abbreviation mapping
# Useful for standardizing/validating company & address name fields
# (e.g., SAP LFA1-NAME1, NAME2, STRAS etc.)

name_abbreviations = {
    # --- Legal / Entity types ---
    "LIMITED": "LTD",
    "INCORPORATED": "INC",
    "CORPORATION": "CORP",
    "COMPANY": "CO",
    "COMPANIES": "COS",
    "ENTERPRISES": "ENTPS",
    "ENTERPRISE": "ENTP",
    "INDUSTRIES": "IND",
    "INDUSTRY": "IND",
    "INDUSTRIAL": "IND",
    "INTERNATIONAL": "INTL",
    "NATIONAL": "NATL",
    "ASSOCIATION": "ASSN",
    "ASSOCIATES": "ASSOC",
    "PARTNERSHIP": "PTNRSHP",
    "PARTNERS": "PTNRS",
    "GROUP": "GRP",
    "HOLDINGS": "HLDGS",
    "HOLDING": "HLDG",
    "SOLUTIONS": "SOLNS",
    "SOLUTION": "SOLN",
    "SERVICES": "SVCS",
    "SERVICE": "SVC",
    "SYSTEMS": "SYS",
    "SYSTEM": "SYS",
    "TECHNOLOGIES": "TECH",
    "TECHNOLOGY": "TECH",
    "MANUFACTURING": "MFG",
    "MANUFACTURER": "MFR",
    "DISTRIBUTION": "DISTR",
    "DISTRIBUTORS": "DISTR",
    "TRADING": "TRDG",
    "COMMERCIAL": "COML",
    "PRIVATE": "PVT",
    "PUBLIC": "PUB",
    "GOVERNMENT": "GOVT",
    "DEPARTMENT": "DEPT",
    "DIVISION": "DIV",
    "BRANCH": "BR",
    "SUBSIDIARY": "SUBSID",
    "FOUNDATION": "FDN",
    "INSTITUTE": "INST",
    "INSTITUTION": "INSTN",
    "ORGANIZATION": "ORG",
    "AUTHORITY": "AUTH",
    "AGENCY": "AGCY",
    "BUREAU": "BUR",
    "COMMISSION": "COMMN",
    "COMMITTEE": "CMTE",
    "COOPERATIVE": "COOP",
    "CONSOLIDATED": "CONSOL",
    "COMMUNICATIONS": "COMM",
    "ENGINEERING": "ENGRG",
    "ENGINEER": "ENGR",
    "ELECTRONICS": "ELEC",
    "ELECTRICAL": "ELEC",
    "ELECTRIC": "ELEC",
    "MECHANICAL": "MECH",
    "MECHANICS": "MECH",
    "CONSTRUCTION": "CONSTR",
    "CONTRACTORS": "CONTR",
    "CONTRACTOR": "CONTR",
    "DEVELOPMENT": "DEV",
    "DEVELOPMENTS": "DEVS",
    "MANAGEMENT": "MGMT",
    "MANAGER": "MGR",
    "ADMINISTRATION": "ADMIN",
    "ADMINISTRATIVE": "ADMIN",
    "PRODUCTS": "PRODS",
    "PRODUCT": "PROD",
    "PRODUCTION": "PRODN",
    "RESOURCES": "RES",
    "RESOURCE": "RES",
    "LOGISTICS": "LOG",
    "TRANSPORTATION": "TRANSP",
    "TRANSPORT": "TRANS",
    "WAREHOUSE": "WHSE",
    "WAREHOUSING": "WHSG",
    "EQUIPMENT": "EQUIP",
    "MATERIALS": "MATLS",
    "MATERIAL": "MATL",
    "CONSULTANTS": "CONSULT",
    "CONSULTANCY": "CONSULT",
    "CONSULTING": "CONSULT",
    "FINANCIAL": "FIN",
    "FINANCE": "FIN",
    "INSURANCE": "INS",
    "INVESTMENTS": "INVS",
    "INVESTMENT": "INV",
    "SECURITIES": "SEC",
    "PROPERTIES": "PROP",
    "PROPERTY": "PROP",
    "REAL ESTATE": "RE",
    "BUILDING": "BLDG",
    "BUILDINGS": "BLDGS",
    "MARKETING": "MKTG",
    "MARKET": "MKT",
    "PROFESSIONAL": "PROF",
    "LABORATORY": "LAB",
    "LABORATORIES": "LABS",
    "RESEARCH": "RES",
    "MEDICAL": "MED",
    "PHARMACEUTICAL": "PHARM",
    "PHARMACEUTICALS": "PHARM",
    "AUTOMOTIVE": "AUTO",
    "AGRICULTURE": "AGRI",
    "AGRICULTURAL": "AGRI"}

In [73]:

legal_suffixes = [
    # United States
    "LLC", "LLP", "LP",
    "INC", "INCORPORATED",
    "CORP", "CORPORATION",
    "CO", "COMPANY",
    "PC", "PLLC",
    "LTD",

    # United Kingdom / Commonwealth
    "LIMITED", "PLC",

    # Germany / DACH
    "GMBH", "GMBH CO KG", "AG", "KG", "OHG", "EK", "UG",

    # France
    "SA", "SARL", "SAS", "SASU", "EURL",

    # Italy
    "SRL", "SPA",

    # Spain / Latin America
    "SL", "SLU", "SA DE CV",

    # Netherlands
    "BV", "NV",

    # Nordics
    "AB", "AS", "OY", "OYJ",

    # GCC / Middle East
    "FZE", "FZC", "FZ-LLC", "WLL", "PJSC", "PSC", "SOLE PROPRIETORSHIP",

    # Asia Pacific
    "PTE LTD", "PVT LTD", "SDN BHD",
    "KK", "YK", "PT", "CO LTD",

    # General / other
    "GROUP", "HOLDINGS", "INTERNATIONAL",
]


In [74]:
# Name Standardization

data["STANDARD_NAME"]=data["NAME1"].str.strip().str.upper()
data["STANDARD_NAME"]=data["STANDARD_NAME"].str.replace(r"[.']+",'',regex=True)
data["STANDARD_NAME"]=data["STANDARD_NAME"].str.replace(r"[,()/-]+",' ',regex=True)
data["STANDARD_NAME"]=data["STANDARD_NAME"].str.replace(r'\s+',' ',regex=True)

for word,abb in name_abbreviations.items():
    pattern = rf"\b{word}\b"
    data["STANDARD_NAME"]=data["STANDARD_NAME"].str.replace(pattern,abb,regex=True)

replace_pattern = r"\b(" + "|".join( re.escape(value) for value in legal_suffixes ) + r")\b$"

data["STANDARD_NAME"]=data["STANDARD_NAME"].str.replace(replace_pattern,' ',regex=True)
data["STANDARD_NAME"]=data["STANDARD_NAME"].str.strip()
data["STANDARD_NAME"]=data["STANDARD_NAME"].str.replace(r'\s+',' ',regex=True)


In [102]:
duplicate_data = data[(data["STANDARD_NAME"].duplicated(keep=False)) & (data["STANDARD_NAME"].notna()) & (data["STANDARD_NAME"]!='') ].copy()

duplicate_data.groupby("STANDARD_NAME").ngroup()

duplicate_data["DUPLICATE_GROUP"]=duplicate_data.groupby("STANDARD_NAME").ngroup()

duplicate_data["SAME_STCD1"]= (duplicate_data[["DUPLICATE_GROUP","STCD1"]].duplicated(keep=False)) & (duplicate_data["STCD1"].notna()) & (duplicate_data["STCD1"]!='')


In [103]:
# Duplicate phone

data["STANDARD_TELF1"]=data["TELF1"].str.replace(r"\D+",'',regex=True)

duplicate_data["SAME_TELF1"] = (
    duplicate_data[["DUPLICATE_GROUP", "STANDARD_TELF1"]].duplicated(keep=False)
    & duplicate_data["STANDARD_TELF1"].notna()
    & (duplicate_data["STANDARD_TELF1"] != '')
)


## Fuzzy Match

In [125]:
from rapidfuzz import fuzz
#print(fuzz.ratio("GULF TRADING SERVICES", "SERVICES GULF TRADING"))
#print(fuzz.token_sort_ratio("GULF TRADING", "TRADNG GULF"))

data["BLOCK_KEY"]=data["STANDARD_NAME"].str[:3]

In [179]:
from itertools import combinations
#vendors = ["A", "B", "C"]
#list(combinations(vendors, 2))

In [ ]:
#data.groupby("BLOCK_KEY").size()
block_data = data[data["BLOCK_KEY"].str.strip() == "AL"]
pairs = list(combinations(block_data.index, 2))
fuzzy_candidates = []
for i, j in pairs:
    score = fuzz.token_sort_ratio(data.loc[i,"STANDARD_NAME"],data.loc[j,"STANDARD_NAME"])

35.71428571428571


In [204]:
block_keys = data["BLOCK_KEY"].str.strip().unique()
pairs=[]
for block in block_keys:
    #print(block)
    block_rows = data[data["BLOCK_KEY"].str.strip() == block]
    pairs.extend(list(combinations(block_rows.index, 2)))

pairs

#fuzzy_candidates = []
#for i, j in pairs:
#    score = fuzz.token_sort_ratio(data.loc[i,"STANDARD_NAME"],data.loc[j,"STANDARD_NAME"])

#print(pairs)

[(0, 1), (2, 8), (5, 6), (10, 11)]